In [2]:
import zipfile

with zipfile.ZipFile("ml-latest-small.zip", 'r') as zip_ref:
    zip_ref.extractall("data")

In [3]:
import pandas as pd

ratings = pd.read_csv("data/ml-latest-small/ratings.csv")
movies = pd.read_csv("data/ml-latest-small/movies.csv")

df = ratings.merge(movies, on="movieId")

df.head()

,userId,movieId,rating,timestamp,title,genres
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,1,3,4.0,964981247,Grumpier Old Men (1995),Comedy|Romance
2,1,6,4.0,964982224,Heat (1995),Action|Crime|Thriller
3,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,5.0,964982931,"Usual Suspects, The (1995)",Crime|Mystery|Thriller


In [37]:
df.shape

(100836, 6)

In [4]:
df["genres"] = df["genres"].apply(lambda x: " ".join(x.split("|")))

In [5]:
movies_df = df[["movieId", "title", "genres"]].drop_duplicates()

In [18]:
print(movies_df.head())
print(movies_df.shape)
print(movies_df.duplicated().sum())

   movieId                        title  \
0        1             Toy Story (1995)   
1        3      Grumpier Old Men (1995)   
2        6                  Heat (1995)   
3       47  Seven (a.k.a. Se7en) (1995)   
4       50   Usual Suspects, The (1995)   

                                        genres  
0  Adventure Animation Children Comedy Fantasy  
1                               Comedy Romance  
2                        Action Crime Thriller  
3                             Mystery Thriller  
4                       Crime Mystery Thriller  
(9724, 3)
0


In [6]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer()
genre_matrix = cv.fit_transform(movies_df["genres"])

In [7]:
genre_matrix.shape

(9724, 24)

In [8]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(genre_matrix)

In [22]:
similarity.shape

(9724, 9724)

In [9]:
movie_index = pd.Series(movies_df.index, index=movies_df["title"]).drop_duplicates()

In [24]:
movie_index

,0
title,
Toy Story (1995),0
Grumpier Old Men (1995),1
Heat (1995),2
Seven (a.k.a. Se7en) (1995),3
"Usual Suspects, The (1995)",4
...,...
Bloodmoon (1997),100820
Sympathy for the Underdog (1971),100821
Hazard (2005),100823


In [10]:
def recommend(movie_name, top_n=5):
    if movie_name not in movie_index:
        return f"Movie '{movie_name}' not found in dataset."

    idx = movie_index[movie_name]
    similarity_scores = list(enumerate(similarity[idx]))
    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)
    similarity_scores = similarity_scores[1:top_n+1]

    movie_indices = [i[0] for i in similarity_scores]
    scores = [i[1] for i in similarity_scores]

    result = movies_df[["title", "genres"]].iloc[movie_indices].copy()
    result["similarity_score"] = scores
    return result

In [11]:
recommend("Toy Story (1995)")

,title,genres,similarity_score
927,Toy Story 2 (1999),Adventure Animation Children Comedy Fantasy,1.0
947,"Monsters, Inc. (2001)",Adventure Animation Children Comedy Fantasy,1.0
2714,Antz (1998),Adventure Animation Children Comedy Fantasy,1.0
2962,"Adventures of Rocky and Bullwinkle, The (2000)",Adventure Animation Children Comedy Fantasy,1.0
3127,"Emperor's New Groove, The (2000)",Adventure Animation Children Comedy Fantasy,1.0


In [12]:
#level 2 user based
user_id = 1

user_history = df[df["userId"] == user_id][["title", "rating", "genres"]].sort_values(by="rating", ascending=False)
user_history.head(10)

,title,rating,genres
3,Seven (a.k.a. Se7en) (1995),5.0,Mystery Thriller
4,"Usual Suspects, The (1995)",5.0,Crime Mystery Thriller
6,Bottle Rocket (1996),5.0,Adventure Comedy Crime Romance
13,Dumb & Dumber (Dumb and Dumber) (1994),5.0,Adventure Comedy
11,Billy Madison (1995),5.0,Comedy
10,Desperado (1995),5.0,Action Romance Western
9,Canadian Bacon (1995),5.0,Comedy War
8,Rob Roy (1995),5.0,Action Drama Romance War
35,Pinocchio (1940),5.0,Animation Children Fantasy Musical
31,Tombstone (1993),5.0,Action Drama Western


In [13]:
liked_movies = df[(df["userId"] == user_id) & (df["rating"] >= 4)][["movieId", "title", "rating", "genres"]]
liked_movies

,movieId,title,rating,genres
0,1,Toy Story (1995),4.0,Adventure Animation Children Comedy Fantasy
1,3,Grumpier Old Men (1995),4.0,Comedy Romance
2,6,Heat (1995),4.0,Action Crime Thriller
3,47,Seven (a.k.a. Se7en) (1995),5.0,Mystery Thriller
4,50,"Usual Suspects, The (1995)",5.0,Crime Mystery Thriller
...,...,...,...,...
227,3744,Shaft (2000),4.0,Action Crime Thriller
228,3793,X-Men (2000),5.0,Action Adventure Sci-Fi
229,3809,What About Bob? (1991),4.0,Comedy
230,4006,Transformers: The Movie (1986),4.0,Adventure Animation Children Sci-Fi


In [15]:
liked_titles = liked_movies["title"].tolist()
liked_titles[:10]

['Toy Story (1995)',
 'Grumpier Old Men (1995)',
 'Heat (1995)',
 'Seven (a.k.a. Se7en) (1995)',
 'Usual Suspects, The (1995)',
 'Bottle Rocket (1996)',
 'Braveheart (1995)',
 'Rob Roy (1995)',
 'Canadian Bacon (1995)',
 'Desperado (1995)']

In [16]:
candidate_movies = []

for movie in liked_titles:
    if movie in movie_index:
        idx = movie_index[movie]
        similarity_scores = list(enumerate(similarity[idx]))
        similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)
        similarity_scores = similarity_scores[1:6]   # top 5 similar movies for each liked movie

        for i in similarity_scores:
            movie_idx = i[0]
            score = i[1]
            title = movies_df.iloc[movie_idx]["title"]
            genres = movies_df.iloc[movie_idx]["genres"]

            candidate_movies.append((title, genres, score))

In [17]:

len(candidate_movies)

1000

In [51]:
candidate_movies[:10]

[('Toy Story 2 (1999)',
  'Adventure Animation Children Comedy Fantasy',
  np.float64(0.9999999999999999)),
 ('Monsters, Inc. (2001)',
  'Adventure Animation Children Comedy Fantasy',
  np.float64(0.9999999999999999)),
 ('Antz (1998)',
  'Adventure Animation Children Comedy Fantasy',
  np.float64(0.9999999999999999)),
 ('Adventures of Rocky and Bullwinkle, The (2000)',
  'Adventure Animation Children Comedy Fantasy',
  np.float64(0.9999999999999999)),
 ("Emperor's New Groove, The (2000)",
  'Adventure Animation Children Comedy Fantasy',
  np.float64(0.9999999999999999)),
 ("She's the One (1996)", 'Comedy Romance', np.float64(0.9999999999999998)),
 ('Wedding Singer, The (1998)',
  'Comedy Romance',
  np.float64(0.9999999999999998)),
 ('20 Dates (1998)', 'Comedy Romance', np.float64(0.9999999999999998)),
 ("You've Got Mail (1998)", 'Comedy Romance', np.float64(0.9999999999999998)),
 ('Four Weddings and a Funeral (1994)',
  'Comedy Romance',
  np.float64(0.9999999999999998))]

In [18]:
candidates_df = pd.DataFrame(candidate_movies, columns=["title", "genres", "score"])
candidates_df.head()

,title,genres,score
0,Toy Story 2 (1999),Adventure Animation Children Comedy Fantasy,1.0
1,"Monsters, Inc. (2001)",Adventure Animation Children Comedy Fantasy,1.0
2,Antz (1998),Adventure Animation Children Comedy Fantasy,1.0
3,"Adventures of Rocky and Bullwinkle, The (2000)",Adventure Animation Children Comedy Fantasy,1.0
4,"Emperor's New Groove, The (2000)",Adventure Animation Children Comedy Fantasy,1.0


In [19]:
candidates_df = candidates_df.sort_values(by="score", ascending=False)
candidates_df = candidates_df.drop_duplicates(subset="title", keep="first")

In [20]:
candidates_df.head(10)
candidates_df.shape

(514, 3)

In [21]:
watched_movies = df[df["userId"] == user_id]["title"].tolist()

In [22]:
final_recommendations = candidates_df[~candidates_df["title"].isin(watched_movies)]

In [23]:
final_recommendations.head(10)

,title,genres,score
999,To Be or Not to Be (1942),Comedy Drama War,1.0
998,Charlie Wilson's War (2007),Comedy Drama War,1.0
997,Mister Roberts (1955),Comedy Drama War,1.0
996,"Great Dictator, The (1940)",Comedy Drama War,1.0
979,"Net, The (1995)",Action Crime Thriller,1.0
978,Die Hard: With a Vengeance (1995),Action Crime Thriller,1.0
977,Kill Bill: Vol. 1 (2003),Action Crime Thriller,1.0
24,The Nice Guys (2016),Crime Mystery Thriller,1.0
23,Now You See Me (2013),Crime Mystery Thriller,1.0
22,Dial M for Murder (1954),Crime Mystery Thriller,1.0


In [24]:
top_n_recommendations = final_recommendations.head(10)
top_n_recommendations

,title,genres,score
999,To Be or Not to Be (1942),Comedy Drama War,1.0
998,Charlie Wilson's War (2007),Comedy Drama War,1.0
997,Mister Roberts (1955),Comedy Drama War,1.0
996,"Great Dictator, The (1940)",Comedy Drama War,1.0
979,"Net, The (1995)",Action Crime Thriller,1.0
978,Die Hard: With a Vengeance (1995),Action Crime Thriller,1.0
977,Kill Bill: Vol. 1 (2003),Action Crime Thriller,1.0
24,The Nice Guys (2016),Crime Mystery Thriller,1.0
23,Now You See Me (2013),Crime Mystery Thriller,1.0
22,Dial M for Murder (1954),Crime Mystery Thriller,1.0


In [25]:
###combined all steps together
def recommend_for_user(user_id, top_n=10, min_rating=4):
    liked_movies = df[(df["userId"] == user_id) & (df["rating"] >= min_rating)][["title"]]
    liked_titles = liked_movies["title"].tolist()

    candidate_movies = []

    for movie in liked_titles:
        if movie in movie_index:
            idx = movie_index[movie]
            similarity_scores = list(enumerate(similarity[idx]))
            similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)
            similarity_scores = similarity_scores[1:6]

            for i in similarity_scores:
                movie_idx = i[0]
                score = i[1]
                title = movies_df.iloc[movie_idx]["title"]
                genres = movies_df.iloc[movie_idx]["genres"]

                candidate_movies.append((title, genres, score))

    candidates_df = pd.DataFrame(candidate_movies, columns=["title", "genres", "score"])
    candidates_df = candidates_df.sort_values(by="score", ascending=False)
    candidates_df = candidates_df.drop_duplicates(subset="title", keep="first")

    watched_movies = df[df["userId"] == user_id]["title"].tolist()
    final_recommendations = candidates_df[~candidates_df["title"].isin(watched_movies)]

    return final_recommendations.head(top_n)

In [26]:
recommend_for_user(1)

,title,genres,score
999,To Be or Not to Be (1942),Comedy Drama War,1.0
998,Charlie Wilson's War (2007),Comedy Drama War,1.0
997,Mister Roberts (1955),Comedy Drama War,1.0
996,"Great Dictator, The (1940)",Comedy Drama War,1.0
979,"Net, The (1995)",Action Crime Thriller,1.0
978,Die Hard: With a Vengeance (1995),Action Crime Thriller,1.0
977,Kill Bill: Vol. 1 (2003),Action Crime Thriller,1.0
24,The Nice Guys (2016),Crime Mystery Thriller,1.0
23,Now You See Me (2013),Crime Mystery Thriller,1.0
22,Dial M for Murder (1954),Crime Mystery Thriller,1.0


In [28]:
#level 3 Collabrating filtering
user_item_matrix = df.pivot_table(index="userId", columns="title", values="rating")
user_item_matrix.head()

title,'71 (2014),'Hellboy': The Seeds of Creation (2004),'Round Midnight (1986),'Salem's Lot (2004),'Til There Was You (1997),'Tis the Season for Love (2015),"'burbs, The (1989)",'night Mother (1986),(500) Days of Summer (2009),*batteries not included (1987),...,Zulu (2013),[REC] (2007),[REC]² (2009),[REC]³ 3 Génesis (2012),anohana: The Flower We Saw That Day - The Movie (2013),eXistenZ (1999),xXx (2002),xXx: State of the Union (2005),¡Three Amigos! (1986),À nous la liberté (Freedom for Us) (1931)
userId,,,,,,,,,,,,,,,,,,,,,
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [29]:
user_item_matrix_filled = user_item_matrix.fillna(0)

In [30]:
from sklearn.metrics.pairwise import cosine_similarity

user_similarity = cosine_similarity(user_item_matrix_filled)

In [31]:
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_item_matrix_filled.index,
    columns=user_item_matrix_filled.index
)

In [32]:
target_user = 1

similar_users = user_similarity_df[target_user].sort_values(ascending=False)
similar_users.head(10)

,1
userId,
1,1.000000
266,0.357408
313,0.351562
368,0.345127
57,0.345034
91,0.334727
469,0.330664
39,0.329782
288,0.329700


In [33]:
top_similar_users = similar_users.head(5).index
top_similar_users

Index([1, 266, 313, 368, 57], dtype='int64', name='userId')

In [34]:
similar_users_movies = df[
    (df["userId"].isin(top_similar_users)) & (df["rating"] >= 4)
][["userId", "title", "rating"]]

similar_users_movies.head(20)

,userId,title,rating
0,1,Toy Story (1995),4.0
1,1,Grumpier Old Men (1995),4.0
2,1,Heat (1995),4.0
3,1,Seven (a.k.a. Se7en) (1995),5.0
4,1,"Usual Suspects, The (1995)",5.0
6,1,Bottle Rocket (1996),5.0
7,1,Braveheart (1995),4.0
8,1,Rob Roy (1995),5.0
9,1,Canadian Bacon (1995),5.0
10,1,Desperado (1995),5.0


In [35]:
watched_movies = df[df["userId"] == target_user]["title"].tolist()

In [36]:
unseen_candidate_movies = similar_users_movies[
    ~similar_users_movies["title"].isin(watched_movies)
]

In [37]:
movie_scores = unseen_candidate_movies.groupby("title").agg({
    "rating": ["mean", "count"]
})

In [38]:
movie_scores.columns = ["avg_rating", "count"]
movie_scores.head()

,avg_rating,count
title,,
1984 (Nineteen Eighty-Four) (1984),5.0,1
2 Days in the Valley (1996),4.0,1
2001: A Space Odyssey (1968),5.0,2
2010: The Year We Make Contact (1984),4.0,1
"7th Voyage of Sinbad, The (1958)",5.0,1


In [39]:
movie_scores = movie_scores.sort_values(
    by=["count", "avg_rating"],
    ascending=False
)

In [73]:
movie_scores.head(10)

,avg_rating,count
title,,
Aliens (1986),4.750000,4
Die Hard (1988),4.000000,4
Blade Runner (1982),5.000000,3
"Godfather, The (1972)",5.000000,3
Heathers (1989),4.666667,3
"Hunt for Red October, The (1990)",4.666667,3
"Godfather: Part II, The (1974)",4.333333,3
"Sixth Sense, The (1999)",4.333333,3
Terminator 2: Judgment Day (1991),4.333333,3


In [40]:
def recommend_for_user_content(df, user_id, top_n=10, min_rating=4):
    # Get liked movies of the target user
    liked_movies = df[(df["userId"] == user_id) & (df["rating"] >= min_rating)][["title"]]
    liked_titles = liked_movies["title"].tolist()

    candidate_movies = []

    # Generate similar movies for each liked movie
    for movie in liked_titles:
        if movie in movie_index:
            idx = movie_index[movie]
            similarity_scores = list(enumerate(similarity[idx]))
            similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)
            similarity_scores = similarity_scores[1:6]  # top 5 similar movies

            for i in similarity_scores:
                movie_idx = i[0]
                score = i[1]
                title = movies_df.iloc[movie_idx]["title"]
                candidate_movies.append((title, score))

    # Convert to dataframe
    candidates_df = pd.DataFrame(candidate_movies, columns=["title", "content_score"])

    # Keep highest content score for duplicate titles
    candidates_df = candidates_df.sort_values(by="content_score", ascending=False)
    candidates_df = candidates_df.drop_duplicates(subset="title", keep="first")

    # Remove already watched movies
    watched_movies = df[df["userId"] == user_id]["title"].tolist()
    final_recommendations = candidates_df[~candidates_df["title"].isin(watched_movies)]

    return final_recommendations.head(top_n)

In [41]:
content_recs = recommend_for_user_content(df, user_id=1, top_n=10)
content_recs

,title,content_score
999,To Be or Not to Be (1942),1.0
998,Charlie Wilson's War (2007),1.0
997,Mister Roberts (1955),1.0
996,"Great Dictator, The (1940)",1.0
979,"Net, The (1995)",1.0
978,Die Hard: With a Vengeance (1995),1.0
977,Kill Bill: Vol. 1 (2003),1.0
24,The Nice Guys (2016),1.0
23,Now You See Me (2013),1.0
22,Dial M for Murder (1954),1.0


In [42]:
def recommend_for_user_collab(df, target_user=1, top_n=10, neighbor_count=5, min_rating=4):
    # Build user-item matrix
    user_item_matrix = df.pivot_table(
        index="userId",
        columns="title",
        values="rating"
    )

    # Fill missing ratings with 0
    user_item_matrix_filled = user_item_matrix.fillna(0)

    # Compute user-user similarity
    user_similarity = cosine_similarity(user_item_matrix_filled)

    user_similarity_df = pd.DataFrame(
        user_similarity,
        index=user_item_matrix_filled.index,
        columns=user_item_matrix_filled.index
    )

    # Find similar users
    similar_users = user_similarity_df[target_user].sort_values(ascending=False)
    similar_users = similar_users.drop(target_user)

    top_similar_users = similar_users.head(neighbor_count).index

    # Movies liked by similar users
    similar_users_movies = df[
        (df["userId"].isin(top_similar_users)) & (df["rating"] >= min_rating)
    ][["userId", "title", "rating"]]

    # Remove movies already watched by target user
    watched_movies = df[df["userId"] == target_user]["title"].tolist()

    unseen_candidate_movies = similar_users_movies[
        ~similar_users_movies["title"].isin(watched_movies)
    ]

    # Aggregate collaborative strength
    movie_scores = unseen_candidate_movies.groupby("title").agg({
        "rating": ["mean", "count"]
    })

    movie_scores.columns = ["avg_rating", "count"]

    # Create one final collaborative score
    movie_scores["collab_score"] = movie_scores["avg_rating"] * movie_scores["count"]

    movie_scores = movie_scores.sort_values(by="collab_score", ascending=False)

    return movie_scores[["collab_score"]].head(top_n).reset_index()

In [43]:
collab_recs = recommend_for_user_collab(df, target_user=1, top_n=10)
collab_recs

,title,collab_score
0,Aliens (1986),24.0
1,"Godfather, The (1972)",20.0
2,Die Hard (1988),20.0
3,Blade Runner (1982),20.0
4,"Hunt for Red October, The (1990)",18.5
5,Terminator 2: Judgment Day (1991),17.0
6,"Sixth Sense, The (1999)",17.0
7,"Godfather: Part II, The (1974)",17.0
8,Glory (1989),16.0
9,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),16.0


In [44]:
hybrid_recs = pd.merge(
    content_recs,
    collab_recs,
    on="title",
    how="outer"
)

In [45]:
hybrid_recs["content_score"] = hybrid_recs["content_score"].fillna(0)
hybrid_recs["collab_score"] = hybrid_recs["collab_score"].fillna(0)

In [46]:
hybrid_recs["hybrid_score"] = (
    0.5 * hybrid_recs["content_score"] +
    0.5 * hybrid_recs["collab_score"]
)

In [47]:
hybrid_recs = hybrid_recs.sort_values(
    by="hybrid_score",
    ascending=False
)

In [48]:
hybrid_recs.head(10)

,title,content_score,collab_score,hybrid_score
0,Aliens (1986),0.0,24.0,12.00
1,Blade Runner (1982),0.0,20.0,10.00
7,"Godfather, The (1972)",0.0,20.0,10.00
4,Die Hard (1988),0.0,20.0,10.00
10,"Hunt for Red October, The (1990)",0.0,18.5,9.25
15,"Sixth Sense, The (1999)",0.0,17.0,8.50
16,Terminator 2: Judgment Day (1991),0.0,17.0,8.50
8,"Godfather: Part II, The (1974)",0.0,17.0,8.50
6,Glory (1989),0.0,16.0,8.00
19,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),0.0,16.0,8.00


In [49]:
def min_max_normalize(series):
    if series.max() == series.min():
        return pd.Series([0] * len(series), index=series.index)
    return (series - series.min()) / (series.max() - series.min())

hybrid_recs["content_score_norm"] = min_max_normalize(hybrid_recs["content_score"])
hybrid_recs["collab_score_norm"] = min_max_normalize(hybrid_recs["collab_score"])

hybrid_recs["hybrid_score"] = (
    0.5 * hybrid_recs["content_score_norm"] +
    0.5 * hybrid_recs["collab_score_norm"]
)

hybrid_recs = hybrid_recs.sort_values(by="hybrid_score", ascending=False)
hybrid_recs.head(10)

,title,content_score,collab_score,hybrid_score,content_score_norm,collab_score_norm
0,Aliens (1986),0.0,24.0,0.5,0.0,1.0
13,"Net, The (1995)",1.0,0.0,0.5,1.0,0.0
2,Charlie Wilson's War (2007),1.0,0.0,0.5,1.0,0.0
3,Dial M for Murder (1954),1.0,0.0,0.5,1.0,0.0
17,The Nice Guys (2016),1.0,0.0,0.5,1.0,0.0
18,To Be or Not to Be (1942),1.0,0.0,0.5,1.0,0.0
9,"Great Dictator, The (1940)",1.0,0.0,0.5,1.0,0.0
14,Now You See Me (2013),1.0,0.0,0.5,1.0,0.0
12,Mister Roberts (1955),1.0,0.0,0.5,1.0,0.0
5,Die Hard: With a Vengeance (1995),1.0,0.0,0.5,1.0,0.0


In [50]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

In [51]:
relevant_test = test_df[test_df["rating"] >= 4]
relevant_test.head()

,userId,movieId,rating,timestamp,title,genres
67037,432,77866,4.5,1335139641,Robin Hood (2010),Action Adventure Drama Romance War
6187,42,2987,4.0,996262677,Who Framed Roger Rabbit? (1988),Adventure Animation Children Comedy Crime Fant...
12229,75,1610,4.0,1158989841,"Hunt for Red October, The (1990)",Action Adventure Thriller
7433,51,177,4.0,1230930634,Lord of Illusions (1995),Horror
65098,416,750,4.5,1187495659,Dr. Strangelove or: How I Learned to Stop Worr...,Comedy War


In [52]:
def get_relevant_items(test_df, user_id, min_rating=4):
    return set(
        test_df[
            (test_df["userId"] == user_id) & (test_df["rating"] >= min_rating)
        ]["title"]
    )

In [53]:
def precision_at_k(recommended_items, relevant_items, k=10):
    recommended_k = recommended_items[:k]

    if len(recommended_k) == 0:
        return 0

    hits = len(set(recommended_k) & relevant_items)
    return hits / len(recommended_k)

In [54]:
def recall_at_k(recommended_items, relevant_items, k=10):
    recommended_k = recommended_items[:k]

    if len(relevant_items) == 0:
        return 0

    hits = len(set(recommended_k) & relevant_items)
    return hits / len(relevant_items)

In [55]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

def recommend_for_user_collab_eval(train_df, target_user=1, top_n=10, neighbor_count=5, min_rating=4):
    user_item_matrix = train_df.pivot_table(
        index="userId",
        columns="title",
        values="rating"
    )

    user_item_matrix_filled = user_item_matrix.fillna(0)

    user_similarity = cosine_similarity(user_item_matrix_filled)

    user_similarity_df = pd.DataFrame(
        user_similarity,
        index=user_item_matrix_filled.index,
        columns=user_item_matrix_filled.index
    )

    if target_user not in user_similarity_df.index:
        return []

    similar_users = user_similarity_df[target_user].sort_values(ascending=False).drop(target_user)
    top_similar_users = similar_users.head(neighbor_count).index

    similar_users_movies = train_df[
        (train_df["userId"].isin(top_similar_users)) & (train_df["rating"] >= min_rating)
    ][["userId", "title", "rating"]]

    watched_movies = train_df[train_df["userId"] == target_user]["title"].tolist()

    unseen_candidate_movies = similar_users_movies[
        ~similar_users_movies["title"].isin(watched_movies)
    ]

    if unseen_candidate_movies.empty:
        return []

    movie_scores = unseen_candidate_movies.groupby("title").agg({
        "rating": ["mean", "count"]
    })

    movie_scores.columns = ["avg_rating", "count"]
    movie_scores["collab_score"] = movie_scores["avg_rating"] * movie_scores["count"]

    movie_scores = movie_scores.sort_values(by="collab_score", ascending=False)

    return movie_scores.head(top_n).index.tolist()

In [56]:
user_id = 1

recommended_items = recommend_for_user_collab_eval(train_df, target_user=user_id, top_n=10)
relevant_items = get_relevant_items(test_df, user_id)

print("Recommended:", recommended_items)
print("Relevant:", relevant_items)

print("Precision@10:", precision_at_k(recommended_items, relevant_items, k=10))
print("Recall@10:", recall_at_k(recommended_items, relevant_items, k=10))

Recommended: ['Glory (1989)', 'Indiana Jones and the Temple of Doom (1984)', 'Hunt for Red October, The (1990)', 'Raiders of the Lost Ark (Indiana Jones and the Raiders of the Lost Ark) (1981)', 'Austin Powers: The Spy Who Shagged Me (1999)', 'Trainspotting (1996)', 'Godfather: Part II, The (1974)', 'Big (1988)', 'Die Hard (1988)', 'Dead Poets Society (1989)']
Relevant: {'South Park: Bigger, Longer and Uncut (1999)', 'Indiana Jones and the Temple of Doom (1984)', 'Gladiator (2000)', 'Big (1988)', 'Big Trouble in Little China (1986)', 'Great Mouse Detective, The (1986)', 'Conan the Barbarian (1982)', 'Pinocchio (1940)', 'Young Frankenstein (1974)', 'Dumbo (1941)', 'Enemy of the State (1998)', 'Rob Roy (1995)', 'Jungle Book, The (1967)', 'Lord of the Rings, The (1978)', 'American History X (1998)', 'Blazing Saddles (1974)', 'Man with the Golden Gun, The (1974)', 'All Quiet on the Western Front (1930)', 'Rocky (1976)', 'Alien (1979)', "Gulliver's Travels (1939)", 'Willow (1988)', 'Live an

In [57]:
users = test_df["userId"].unique()

precision_scores = []
recall_scores = []

for user_id in users:
    recommended_items = recommend_for_user_collab_eval(train_df, target_user=user_id, top_n=10)
    relevant_items = get_relevant_items(test_df, user_id)

    if len(relevant_items) > 0:
        precision_scores.append(precision_at_k(recommended_items, relevant_items, k=10))
        recall_scores.append(recall_at_k(recommended_items, relevant_items, k=10))

print("Average Precision@10:", sum(precision_scores) / len(precision_scores))
print("Average Recall@10:", sum(recall_scores) / len(recall_scores))

Average Precision@10: 0.164440734557596
Average Recall@10: 0.16097942786718622
